# C6 example 3/4: `WETensorProduct` with external circular harmonics

This architecture makes the interaction geometry-conditioned. The node still contains three 3D vectors and two scalars, explicitly packed as $3E_1\oplus5A$. In addition, the node/message has a 3D geometric direction $r$. For C6, its $(r_x,r_y)$ plane rotates while $r_z$ is invariant.

`WETensorProduct` expects filter features to be computed externally, so this notebook performs

$$r_{xy}\xrightarrow{\operatorname{atan2}}\theta\xrightarrow{Y_{C_6}}Y(\theta)\xrightarrow{\mathrm{WETP}}\text{features}.$$

The default C6 circular bandlimit is $L_{full}=\lfloor6/2\rfloor=3$. Radius and $r_z$ are invariant inputs to small radial networks that produce the per-sample reduced weights. The radial-network parameters are shared over all nodes/messages, although their numerical outputs may depend on geometry.

In [ ]:
import torch
from we3nn import CircularHarmonics, CyclicGroup, nn

torch.manual_seed(7)
torch.set_printoptions(precision=5, sci_mode=False)
G = CyclicGroup(6)
A = G.trivial_representation
E1 = G.standard_representation
regular = G.regular_representation()
input_rep = 3 * E1 + 5 * A
hidden_rep = 2 * regular
output_rep = E1 + 4 * A
harmonics = CircularHarmonics(G)  # max_frequency=None -> floor(6/2)=3
filter_rep = harmonics.rep_out
print('harmonic frequencies: 0..', harmonics.max_frequency)
print('filter representation:', filter_rep.name)
print('dimensions:', input_rep.size, 'x', filter_rep.size, '->', hidden_rep.size, '->', output_rep.size)

In [ ]:
def pack_input(vectors, scalars):
    xy = vectors[..., :, :2].reshape(*vectors.shape[:-2], 6)
    return torch.cat((xy, vectors[..., :, 2], scalars), dim=-1)

def unpack_input(x):
    xy = x[..., :6].reshape(*x.shape[:-1], 3, 2)
    return torch.cat((xy, x[..., 6:9].unsqueeze(-1)), dim=-1), x[..., 9:11]

def unpack_output(y):
    return torch.cat((y[..., :2], y[..., 2:3]), dim=-1), y[..., 3:6]

def rotate_points(points, element):
    Rxy = E1(element).to(device=points.device, dtype=points.dtype)
    xy = points[..., :2] @ Rxy.T
    return torch.cat((xy, points[..., 2:3]), dim=-1)

vectors = torch.tensor([[[1.0, 0.2, -0.4], [-0.3, 0.8, 1.2], [0.5, -0.7, 0.1]]])
scalars = torch.tensor([[0.6, -1.1]])
point = torch.tensor([[0.8, 0.35, -0.2]])
x = nn.RepresentationTensor(pack_input(vectors, scalars), input_rep)

angle = torch.atan2(point[..., 1], point[..., 0])
Y = harmonics(angle)  # equivalently: harmonics.from_vectors(point[..., :2])
print('point:', point)
print('angle:', angle)
print('external circular harmonics Y(theta):', Y)

## Architecture

For each layer, the finite-group coupling tensors $C_p$ are fixed, the harmonic filter $Y_j(\theta)$ carries angular dependence, and an invariant radial network supplies reduced coefficients $w_p(\lVert r_{xy}\rVert,r_z)$:

$$z_o=\sum_p w_p(\lVert r_{xy}\rVert,r_z)(C_p)_{oij}x_iY_j(\theta).$$

The hidden regular representations admit coordinatewise `PointActiv`. Both Wigner--Eckart layers reuse the same externally computed harmonic sample but have separate radial networks.

In [ ]:
class ExternalHarmonicNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.harmonics = harmonics
        self.input_layer = nn.WETensorProduct(
            input_rep, filter_rep, hidden_rep, shared_weights=False
        )
        self.activation = nn.PointActiv(hidden_rep, torch.relu)
        self.output_layer = nn.WETensorProduct(
            hidden_rep, filter_rep, output_rep, shared_weights=False
        )
        # These networks are shared across samples. Their outputs are the
        # external reduced coefficients expected by WETensorProduct.
        self.radial_in = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.input_layer.weight_numel),
        )
        self.radial_out = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.output_layer.weight_numel),
        )

    def forward(self, features, points):
        theta = torch.atan2(points[..., 1], points[..., 0])
        filter_features = nn.RepresentationTensor(self.harmonics(theta), filter_rep)
        invariant_geometry = torch.stack(
            (torch.linalg.vector_norm(points[..., :2], dim=-1), points[..., 2]),
            dim=-1,
        )
        w_in = self.radial_in(invariant_geometry)
        w_out = self.radial_out(invariant_geometry)
        h_pre = self.input_layer(features, filter_features, w_in)
        h = self.activation(h_pre)
        y = self.output_layer(h, filter_features, w_out)
        return y, filter_features, w_in, w_out, h_pre, h

model = ExternalHarmonicNetwork().eval()
y, Y_typed, w_in, w_out, h_pre, h = model(x, point)
print('input/output reduced-weight counts:', model.input_layer.weight_numel, model.output_layer.weight_numel)
print('hidden before PointActiv:', h_pre.tensor)
print('hidden after  PointActiv:', h.tensor)
print('physical output (vector, scalars):', unpack_output(y.tensor))
kernel_basis = model.input_layer.sample_kernel_basis(Y_typed)
print('sampled input-kernel basis shape [batch, paths, out, in]:', tuple(kernel_basis.shape))

## All six simultaneous rotations

Equivariance requires rotating both the node features and the geometric point. The harmonic values generally change, while radial weights remain identical because their inputs are invariant.

In [ ]:
errors = []
for k, element in enumerate(G.elements):
    x_k = x.transform_fibers(element)
    point_k = rotate_points(point, element)
    y_k, Y_k, w_in_k, w_out_k, _, _ = model(x_k, point_k)
    expected_k = y.transform_fibers(element)
    error = (y_k.tensor - expected_k.tensor).abs().max().item()
    errors.append(error)
    torch.testing.assert_close(y_k.tensor, expected_k.tensor, atol=5e-5, rtol=5e-5)
    torch.testing.assert_close(w_in_k, w_in, atol=1e-6, rtol=1e-6)
    torch.testing.assert_close(w_out_k, w_out, atol=1e-6, rtol=1e-6)
    in_vectors_k, in_scalars_k = unpack_input(x_k.tensor)
    out_vector_k, out_scalars_k = unpack_output(y_k.tensor)
    print(f'rotation {k}: angle={60*k:3d} degrees')
    print('  rotated geometry:', point_k[0].tolist())
    print('  circular Y      :', Y_k.tensor[0].tolist())
    print('  input vectors   :', in_vectors_k[0].tolist())
    print('  input scalars   :', in_scalars_k[0].tolist())
    print('  output vector   :', out_vector_k[0].tolist())
    print('  output scalars  :', out_scalars_k[0].tolist())
    print(f'  max equivariance error: {error:.3e}')

print('maximum over all rotations:', max(errors))

## Visualizing geometry, harmonics, regular features, and kernels

The circular-harmonic panel names every component used by the full C6 bandlimit: frequency 0, the cosine/sine pairs at frequencies 1 and 2, and the two one-dimensional Nyquist components at frequency 3. The effective-kernel panel displays $K(r)=\sum_p w_p(r)K_p(r)$ for the first Wigner--Eckart layer, so the hidden pre-activation is literally $K(r)x$.

In [ ]:
import matplotlib.pyplot as plt

hidden_by_rotation, harmonic_by_rotation = [], []
output_vectors, output_scalars = [], []
for element in G.elements:
    x_k = x.transform_fibers(element)
    point_k = rotate_points(point, element)
    y_k, Y_k, _, _, _, h_k = model(x_k, point_k)
    vector_k, scalars_k = unpack_output(y_k.tensor)
    hidden_by_rotation.append(h_k.tensor[0].detach())
    harmonic_by_rotation.append(Y_k.tensor[0].detach())
    output_vectors.append(vector_k[0].detach())
    output_scalars.append(scalars_k[0].detach())
hidden_by_rotation = torch.stack(hidden_by_rotation).cpu()
harmonic_by_rotation = torch.stack(harmonic_by_rotation).cpu()
output_vectors = torch.stack(output_vectors).cpu()
output_scalars = torch.stack(output_scalars).cpu()
angles_deg = torch.arange(6) * 60
colors = plt.cm.hsv(torch.linspace(0, 5/6, 6).numpy())

harmonic_labels = []
for frequency, mode in harmonics._layout:
    if mode == 'pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'sin({frequency}θ)'))
    elif mode == 'conjugate_pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'-sin({frequency}θ)'))
    else:
        harmonic_labels.append(f'{mode}({frequency}θ)')

# sample_kernel_basis returns K_p(r); radial weights contract the path axis.
basis_at_point = model.input_layer.sample_kernel_basis(Y_typed)[0].detach()
effective_kernel = torch.einsum('p,poi->oi', w_in[0].detach(), basis_at_point).cpu()
torch.testing.assert_close(h_pre.tensor[0], effective_kernel @ x.tensor[0], atol=2e-5, rtol=2e-5)

fig = plt.figure(figsize=(18, 10), constrained_layout=True)
ax_in = fig.add_subplot(2, 3, 1, projection='3d')
for index, vector in enumerate(vectors[0]):
    ax_in.quiver(0, 0, 0, *vector.tolist(), color=f'C{index}', linewidth=2, label=f'input v{index+1}')
ax_in.quiver(0, 0, 0, *point[0].tolist(), color='black', linestyle='--', linewidth=2, label='geometry r')
ax_in.set(xlabel='x', ylabel='y', zlabel='z', title='Initial vectors and geometry')
setup_limit = 1.15 * torch.cat((vectors[0], point)).abs().max().item()
ax_in.set_xlim(-setup_limit, setup_limit); ax_in.set_ylim(-setup_limit, setup_limit); ax_in.set_zlim(-setup_limit, setup_limit); ax_in.set_box_aspect((1, 1, 1))
ax_in.legend(fontsize=8)

ax_harm = fig.add_subplot(2, 3, 2)
for component, label in enumerate(harmonic_labels):
    ax_harm.plot(angles_deg, harmonic_by_rotation[:, component], marker='o', label=label)
ax_harm.set(xticks=angles_deg.tolist(), xlabel='C6 rotation', ylabel='harmonic value', title='Circular harmonics: frequencies 0, 1, 2, 3')
ax_harm.grid(alpha=0.3); ax_harm.legend(fontsize=7, ncols=2)

ax_hidden = fig.add_subplot(2, 3, 3)
image = ax_hidden.imshow(hidden_by_rotation, aspect='auto', cmap='coolwarm')
ax_hidden.axvline(5.5, color='white', linewidth=2)
ax_hidden.set(xticks=range(12), yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='regular coordinate (copies 1 | 2)', ylabel='rotation', title='Hidden 2 Reg(C6) after PointActiv')
fig.colorbar(image, ax=ax_hidden, shrink=0.75)

ax_kernel = fig.add_subplot(2, 3, 4)
kernel_image = ax_kernel.imshow(effective_kernel, aspect='auto', cmap='coolwarm')
ax_kernel.set(xlabel='input coordinate', ylabel='hidden coordinate', title='Effective input-layer kernel K(r)')
fig.colorbar(kernel_image, ax=ax_kernel, shrink=0.75)

ax_vec = fig.add_subplot(2, 3, 5, projection='3d')
for k, (vector, color) in enumerate(zip(output_vectors, colors)):
    ax_vec.quiver(0, 0, 0, *vector.tolist(), color=color, linewidth=2, label=f'{60*k}°')
ax_vec.set(xlabel='x', ylabel='y', zlabel='z', title='WETensorProduct output vector')
output_limit = max(1e-3, 1.15 * output_vectors.abs().max().item())
ax_vec.set_xlim(-output_limit, output_limit); ax_vec.set_ylim(-output_limit, output_limit); ax_vec.set_zlim(-output_limit, output_limit); ax_vec.set_box_aspect((1, 1, 1))
ax_vec.legend(ncols=2, fontsize=7)

ax_scalar = fig.add_subplot(2, 3, 6)
for channel in range(3):
    ax_scalar.plot(angles_deg, output_scalars[:, channel], marker='o', label=f'output scalar {channel+1}')
ax_scalar.set(xticks=angles_deg.tolist(), xlabel='C6 rotation', ylabel='value', title='WETensorProduct output scalars')
ax_scalar.grid(alpha=0.3); ax_scalar.legend()
plt.show()